# Modern Coding Practice — Itinerary Reconstruction (Amazon FAR style)

The boarding-pass problem: an old chest holds a pile of passes, each just `(source, destination)`.
Reconstruct the trip. Parts 1-3 build the core (concurrency at Part 3); Parts 4-5 are stretch tasks
(messy/repeated tickets via Eulerian reconstruction, then reconstructing many chests in parallel).
Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only after trying.

---

## Part 1 — Reconstruct a simple chain

`reconstruct(cards) -> list[str]`. The pile forms a single simple path: each city is the source of
at most one pass and the destination of at most one pass, with exactly one start (a source that is
never a destination). Return the ordered list of cities.

**Lock down:** order of the pile is arbitrary (sort/lookup, don't assume). Find the start by degree
(the city that is never a destination), then follow successors. O(n) with hash maps.

In [ ]:
def reconstruct(cards):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    cards = [("JFK", "LAX"), ("SFO", "JFK"), ("LAX", "ORD")]
    assert reconstruct(cards) == ["SFO", "JFK", "LAX", "ORD"]
    assert reconstruct([("A", "B")]) == ["A", "B"]
    import random
    pile = cards[:]
    random.Random(1).shuffle(pile)
    assert reconstruct(pile) == ["SFO", "JFK", "LAX", "ORD"]   # pile order must not matter
    print("Part 1 OK")

_t1()

## Part 2 — Is the route unique?

`is_unique(cards) -> bool`: do the passes determine **exactly one** valid itinerary using all of
them? This is the "how do we know the route is unique" question. Return `False` when the answer is
ambiguous or invalid.

**The conditions to articulate:**
- No branching: every city has out-degree ≤ 1 and in-degree ≤ 1 (two passes out of one city ⇒ two
  possible routes).
- Exactly one start (in 0 / out 1) and one end (in 1 / out 0).
- Following the chain from the start consumes **all** passes (single connected component, no stray
  cycle, no duplicates).

In [ ]:
def is_unique(cards) -> bool:
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    assert is_unique([("SFO", "JFK"), ("JFK", "LAX"), ("LAX", "ORD")]) is True
    assert is_unique([("A", "B")]) is True
    assert is_unique([("SFO", "JFK"), ("JFK", "LAX"), ("JFK", "ORD")]) is False  # branching
    assert is_unique([("A", "B"), ("C", "D")]) is False                          # disconnected
    assert is_unique([("A", "B"), ("B", "A")]) is False                          # cycle, no start
    assert is_unique([("A", "B"), ("A", "B")]) is False                          # duplicate pass
    assert is_unique([("A", "B"), ("C", "D"), ("D", "C")]) is False              # path + stray cycle
    assert is_unique([]) is False
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent ingestion from many chests

Several searchers empty different chests at once into one shared `ItineraryBuilder`. Make `add(card)`
thread-safe; `reconstruct()` returns the route; `count()` returns passes added so far.

**The point:** `add` does a compound update (append the pass *and* bump a counter). 100 threads
adding concurrently must leave `count() == total` — an unguarded `count += 1` is a lost-update race
(load/add/store). **Discuss:** that `list.append` happens to be atomic under the GIL but the
read-modify-write counter is not, so you still need the lock; this is coordination, not CPU work.

In [ ]:
import threading


class ItineraryBuilder:
    def __init__(self):
        raise NotImplementedError

    def add(self, card):
        raise NotImplementedError

    def count(self):
        raise NotImplementedError

    def reconstruct(self):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    cards = [("C%d" % i, "C%d" % (i + 1)) for i in range(200)]   # chain C0 -> ... -> C200
    builder = ItineraryBuilder()
    import random
    pile = cards[:]
    random.Random(0).shuffle(pile)
    parts = [pile[i::8] for i in range(8)]
    barrier = threading.Barrier(8)

    def worker(part):
        barrier.wait()
        for c in part:
            builder.add(c)

    ts = [threading.Thread(target=worker, args=(p,)) for p in parts]
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    assert builder.count() == 200                      # no lost updates
    assert builder.reconstruct() == ["C%d" % i for i in range(201)]
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Messy chests: repeated tickets (Eulerian)

Real chests are messy: a city can be visited more than once and the same route can appear multiple
times. The clean "each city once" assumption breaks. Now you must use **every** pass exactly once,
which is an Eulerian-path problem.

`reconstruct_euler(cards) -> list[str] | None`:
- Use all passes exactly once; cities may repeat.
- Among valid itineraries, return the **lexicographically smallest** (Hierholzer's algorithm with
  sorted adjacency).
- Pick the start by degree (the city with out−in = 1), or, for a balanced circuit, the smallest city
  with an outgoing pass.
- Return `None` when no Eulerian path uses all passes (disconnected or bad degrees).

**Discuss:** Eulerian path existence (degree + connectivity), why Hierholzer is O(E) vs naive
backtracking, and how lexicographic order falls out of a sorted adjacency + post-order reversal.

In [ ]:
import heapq
from collections import defaultdict


def reconstruct_euler(cards):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    assert reconstruct_euler([("JFK", "ATL"), ("ATL", "JFK"), ("JFK", "SFO")]) == \
        ["JFK", "ATL", "JFK", "SFO"]
    assert reconstruct_euler([("JFK", "KUL"), ("JFK", "NRT"), ("NRT", "JFK")]) == \
        ["JFK", "NRT", "JFK", "KUL"]                 # lexicographically smallest valid route
    assert reconstruct_euler([("A", "B"), ("B", "C"), ("C", "A")]) == ["A", "B", "C", "A"]  # circuit
    assert reconstruct_euler([("A", "B"), ("C", "D")]) is None     # disconnected -> impossible
    assert reconstruct_euler([]) == []
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Reconstruct many chests in parallel

A warehouse of chests, each an independent trip (possibly Eulerian). `parallel_reconstruct(chests)
-> dict[name, route]` reconstructs them all at once across processes with `ProcessPoolExecutor`;
`chests` is `dict[name, list[cards]]` and the worker is `itinerary_workers.reconstruct_one`. A chest
that can't be reconstructed maps to `None`.

**Discuss:** each chest is independent (embarrassingly parallel), GIL (reconstruction is CPU-bound ->
processes), pickling the card lists, and load-balancing many small vs few huge chests.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import itinerary_workers


def parallel_reconstruct(chests) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    chests = {
        "trip1": [("SFO", "JFK"), ("JFK", "LAX")],
        "trip2": [("A", "B"), ("B", "C"), ("C", "A")],
        "broken": [("X", "Y"), ("Z", "W")],
    }
    res = parallel_reconstruct(chests)
    assert res["trip1"] == ["SFO", "JFK", "LAX"]
    assert res["trip2"] == ["A", "B", "C", "A"]
    assert res["broken"] is None
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Count how many distinct valid itineraries exist (not just one).
- Passes with timestamps/flight numbers: validate temporal feasibility, layover constraints.
- Multiple travelers interleaved in one pile: partition into per-traveler itineraries first.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import heapq
import threading
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
import itinerary_workers


def ref_reconstruct(cards):
    succ = {s: d for s, d in cards}
    dests = {d for _, d in cards}
    start = next(s for s, _ in cards if s not in dests)
    route = [start]
    while route[-1] in succ:
        route.append(succ[route[-1]])
    return route


def ref_is_unique(cards):
    if not cards or len(set(cards)) != len(cards):     # empty or duplicate pass
        return False
    out_deg, in_deg, succ, cities = {}, {}, {}, set()
    for s, d in cards:
        out_deg[s] = out_deg.get(s, 0) + 1
        in_deg[d] = in_deg.get(d, 0) + 1
        succ[s] = d
        cities.update((s, d))
    if any(v > 1 for v in out_deg.values()) or any(v > 1 for v in in_deg.values()):
        return False                                   # branching
    starts = [c for c in cities if out_deg.get(c, 0) == 1 and in_deg.get(c, 0) == 0]
    ends = [c for c in cities if in_deg.get(c, 0) == 1 and out_deg.get(c, 0) == 0]
    if len(starts) != 1 or len(ends) != 1:
        return False
    cur, used = starts[0], 0
    while cur in succ:
        cur = succ[cur]
        used += 1
    return used == len(cards)                           # all passes consumed in one chain


class RefBuilder:
    def __init__(self):
        self._cards = []
        self._count = 0
        self._lock = threading.Lock()

    def add(self, card):
        with self._lock:                                # guard the compound update
            self._cards.append(card)
            self._count += 1

    def count(self):
        with self._lock:
            return self._count

    def reconstruct(self):
        with self._lock:
            cards = list(self._cards)
        return ref_reconstruct(cards)


def ref_reconstruct_euler(cards):
    if not cards:
        return []
    graph = defaultdict(list)
    out_deg, in_deg = defaultdict(int), defaultdict(int)
    for s, d in cards:
        graph[s].append(d)
        out_deg[s] += 1
        in_deg[d] += 1
    for s in graph:
        graph[s].sort(reverse=True)                     # pop() yields smallest

    vertices = set(out_deg) | set(in_deg)
    starts = [v for v in vertices if out_deg[v] - in_deg[v] == 1]
    ends = [v for v in vertices if in_deg[v] - out_deg[v] == 1]
    if len(starts) > 1 or len(ends) > 1:
        return None
    if len(starts) == 1:
        start = starts[0]
    else:
        if ends:
            return None
        start = min(v for v in vertices if out_deg[v] > 0)

    local = {k: list(v) for k, v in graph.items()}
    route, stack = [], [start]
    while stack:
        v = stack[-1]
        if local.get(v):
            stack.append(local[v].pop())
        else:
            route.append(stack.pop())
    route.reverse()
    if len(route) - 1 != len(cards):
        return None
    return route


def ref_parallel_reconstruct(chests):
    items = list(chests.items())
    with ProcessPoolExecutor() as ex:
        return dict(ex.map(itinerary_workers.reconstruct_one, items))


assert ref_reconstruct([("B", "C"), ("A", "B")]) == ["A", "B", "C"]
assert ref_is_unique([("A", "B"), ("B", "C")]) is True
assert ref_is_unique([("A", "B"), ("A", "C")]) is False
assert ref_reconstruct_euler([("JFK", "ATL"), ("ATL", "JFK"), ("JFK", "SFO")]) == ["JFK", "ATL", "JFK", "SFO"]
assert ref_parallel_reconstruct({"t": [("A", "B"), ("B", "C")]}) == {"t": ["A", "B", "C"]}
print("reference solutions OK")